# 2. Fine-tuning Qwen2-VL on the manga task (TRL)

Following the Hugging Face cookbook "Fine-Tuning a Vision Language Model (Qwen2-VL-7B) with the Hugging Face
Ecosystem (TRL)": same model, 4-bit QLoRA, same LoRA settings, `SFTTrainer`.
Only the given dataset (no PopManga).

input: the 3 pages of a sequence. output: every story line in reading order as `speaker: text`.

Kaggle: GPU T4 x2 (only one is used), internet on, task data as input.
Changes from the cookbook: T4 has no bf16 so fp16 is used, one GPU only (newer TRL refuses two in a
notebook), pages are made smaller so 3 of them fit in memory, batch size 1, the model is tested in 4-bit,
and no hub upload / trackio.

In [1]:
!pip install -U -q trl bitsandbytes peft qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 11.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 49.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 52.4 MB/s eta 0:00:00:00:0100:01


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"     # use one GPU only (TRL refuses to run when it sees two)

import re
import json
import random
import torch
from PIL import Image

DATA = None
for root, dirs, files in os.walk("/kaggle/input", followlinks=True):
    if "sequences.json" in files:
        DATA = root
print(DATA)

/kaggle/input/datasets/arpitparashar06/manga-task/manga-task-upload/manga-task/dataset


## data

In [3]:
system_message = """You are a Vision Language Model that transcribes manga.
You get 3 consecutive manga pages. Write every story line (speech, thoughts, narration) in reading order, page by page.
Start every page with a line "PAGE n". Write each line as "speaker: text".
The speaker is character1, character2, ... (the same character keeps the same name on all 3 pages) or NARRATION.
Skip sound effects."""

In [4]:
images = {}
for s in json.load(open(DATA + "/sequences.json")):
    images[s["sequence_id"]] = [DATA + "/" + p for p in s["images"]]

gold = {}
for line in open(DATA + "/development/labels.jsonl"):
    row = json.loads(line)
    gold[row["sequence_id"]] = row["pages"]

def load_page(path):
    # smaller pages so 3 of them fit on a T4
    img = Image.open(path).convert("RGB")
    img.thumbnail((640, 640))
    return img

def answer_text(pages):
    out = []
    for p in range(3):
        out.append("PAGE " + str(p + 1))
        for x in pages[p]:
            out.append(x["speaker"] + ": " + x["text"])
    return "\n".join(out)

print(answer_text(gold[list(gold)[0]])[:500])

PAGE 1
character1: HUH? DOES KOMABA SEEM DIFFERENT THAN USUAL?
character2: HE'S DOING REALLY WELL. IT LOOKS LIKE HE'S BEEN WORKING HARD FOR THE FALL TOURNAMENT.
UNKNOWN: THAT'S THE WAY!
character1: NO, THAT'S NOT IT. IT'S LIKE HE'S...
character1: OUT FOR BLOOD.
PAGE 2
PAGE 3
UNKNOWN: WOAH!!
character3: OKAWA-SENPAI, YOU LOOK SO COOL!
character4: RIGHT HERE! THIS IS WHERE IT HAPPENS!
UNKNOWN: SO CLOSE!!
character5: DO YOU HAVE TO KEEP WATCHING THAT? IT'S EMBARRASSING.
character4: IT'S ALL FOR THE


In [5]:
def format_data(sid):
    pages = [load_page(p) for p in images[sid]]
    # one {"type": "image"} per page, the pictures themselves go in "images"
    user_content = []
    for img in pages:
        user_content.append({"type": "image"})
    user_content.append({"type": "text", "text": "Transcribe these 3 pages."})

    messages = [
        {"role": "system", "content": [{"type": "text", "text": system_message}]},
        {"role": "user", "content": user_content},
    ]
    if sid in gold:
        messages.append({"role": "assistant", "content": [{"type": "text", "text": answer_text(gold[sid])}]})
    return {"images": pages, "messages": messages}

In [6]:
# same split as my other experiments: 16 dev sequences for validation, 64 for training
ids = list(gold.keys())
random.seed(42)
random.shuffle(ids)
val_ids = ids[:16]
train_ids = ids[16:]
test_ids = [s for s in images if s not in gold]

from datasets import Dataset

train_dataset = Dataset.from_list([format_data(s) for s in train_ids])
eval_dataset = Dataset.from_list([format_data(s) for s in val_ids])
print(len(train_dataset), "train,", len(eval_dataset), "val,", len(test_ids), "test")

64 train, 16 val, 15 test


## model (4-bit, like the cookbook)

In [7]:
from transformers import Qwen2VLForConditionalGeneration, Qwen2VLProcessor, BitsAndBytesConfig

model_id = "Qwen/Qwen2-VL-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,      # cookbook uses bfloat16, T4 can't
)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    device_map={"": 0},          # all on GPU 0 (split over two GPUs breaks the loss step)
    torch_dtype=torch.float16,
    quantization_config=bnb_config,
)
processor = Qwen2VLProcessor.from_pretrained(model_id)

config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/56.5k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

...

preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [8]:
def generate_text_from_sample(model, processor, sample, max_new_tokens=1500, device="cuda"):
    text_input = processor.apply_chat_template(sample["messages"][:2], tokenize=False, add_generation_prompt=True)
    model_inputs = processor(text=[text_input], images=sample["images"], return_tensors="pt").to(device)

    generated_ids = model.generate(**model_inputs, max_new_tokens=max_new_tokens)
    trimmed = [out[len(inp):] for inp, out in zip(model_inputs.input_ids, generated_ids)]
    return processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

# before training
print(generate_text_from_sample(model, processor, eval_dataset[0])[:800])

PAGE 26: "The result of a curse... The majority are the result of negative energy that flows out of people... Places where crows gather, like schools or hospitals... A curse?"

PAGE 27: "A curse? It doesn't matter if you believe me or not! Anyway..."

PAGE 28: "The object you found is such an item. For that reason, the cursed object is often found at such places as a talisman. Pain, regret, humiliation... The more people dwell on bad memories... The more these places become receptive to those feelings... A talisman?"

PAGE 29: "Using poison to fend off poison. Then it attracts curses and feeds on them. After many years, the seal weakens and the talisman essentially changes form. Compared to other cursed objects, the one placed at your school is extremely dangerous and is categorized as spe


## LoRA + training

In [9]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=8,
    bias="none",
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM",
)

training_args = SFTConfig(
    output_dir="qwen2-vl-manga",
    num_train_epochs=3,
    per_device_train_batch_size=1,          # cookbook: 4 (3 pages per sample is a lot more memory)
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_length=None,
    optim="adamw_torch_fused",
    learning_rate=2e-4,
    logging_steps=5,
    eval_steps=10,
    eval_strategy="steps",
    save_strategy="steps",
    save_steps=20,
    fp16=True,                               # cookbook: bf16=True
    max_grad_norm=0.3,
    warmup_steps=1,                          # cookbook: warmup_ratio=0.03 (removed in new transformers)
    push_to_hub=False,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=peft_config,
    processing_class=processor,
)

# fp16 training needs the trainable (LoRA) weights in float32, the rest stays 4-bit
for p in trainer.model.parameters():
    if p.requires_grad:
        p.data = p.data.float()

In [10]:
trainer.train()
trainer.save_model(training_args.output_dir)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
10,12.833376,12.774914,4.615062,0.184471,121595.000000
20,11.857164,11.724469,5.724083,0.185792,242260.000000
24,11.857164,11.584143,5.853076,0.185668,291900.000000


## test the fine-tuned model

In [11]:
import gc
import time

def clear_memory():
    for name in ["inputs", "model", "processor", "trainer"]:
        if name in globals():
            del globals()[name]
    time.sleep(2)
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    print(f"GPU allocated memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

clear_memory()

# 4-bit again: the full-size model does not fit on one T4
model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id, device_map={"": 0}, torch_dtype=torch.float16, quantization_config=bnb_config)
processor = Qwen2VLProcessor.from_pretrained(model_id)
model.load_adapter("qwen2-vl-manga")

GPU allocated memory: 1.03 GB


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/112 [00:00<?, ?it/s]

LoadStateDictInfo(missing_keys=set(), unexpected_keys=set(), mismatched_keys=set(), error_msgs=[], conversion_errors={}, skipped_pp_keys=set())

In [12]:
# model text -> the task's json format
def parse(text):
    pages = [[], [], []]
    p = 0
    for line in text.split("\n"):
        line = line.strip()
        m = re.match(r"PAGE\s*(\d)", line)
        if m:
            p = min(max(int(m.group(1)) - 1, 0), 2)
            continue
        if ":" not in line:
            continue
        speaker, words = line.split(":", 1)
        speaker = speaker.strip()
        words = words.strip()
        if words == "":
            continue
        if speaker != "NARRATION" and not re.fullmatch(r"character\d+", speaker):
            speaker = "UNKNOWN"
        pages[p].append({"speaker": speaker, "text": words})

    # UNKNOWN is only for the gold labels, never predict it: give it the previous speaker
    last = "character1"
    for page in pages:
        for x in page:
            if x["speaker"] == "UNKNOWN":
                x["speaker"] = last
            last = x["speaker"]
    return pages

out = generate_text_from_sample(model, processor, eval_dataset[0])
print(out[:800])
print(parse(out)[0][:5])

PAGE 1
NARRATION: The result of a curse...
NARRATION: Places where crows gather, like schools or hospitals...
NARRATION: The majority are the result of negative energy that flows out of people...
NARRATION: Are susceptible to receiving that energy...
NARRATION: The object you found is such an item.
NARRATION: For that reason, it is often found at such places as a talisman.
NARRATION: The more these places become receptive to those feelings...
NARRATION: Pain, regret, humiliation... the more people dwell on bad memories...
NARRATION: The more these places become receptive to those feelings...
NARRATION: A talisman?

PAGE 2
NARRATION: The more evil placed on a cursed object, the more it deflects other curses.
NARRATION: Then it attracts curses and feeds on them.
NARRATION: After many years, 
[{'speaker': 'NARRATION', 'text': 'The result of a curse...'}, {'speaker': 'NARRATION', 'text': 'Places where crows gather, like schools or hospitals...'}, {'speaker': 'NARRATION', 'text': 'The major

In [13]:
# all 16 validation sequences
f = open("/kaggle/working/val_labels.jsonl", "w")
for sid in val_ids:
    f.write(json.dumps({"sequence_id": sid, "pages": gold[sid]}) + "\n")
f.close()

f = open("/kaggle/working/qwen_val.jsonl", "w")
for i, sid in enumerate(val_ids):
    out = generate_text_from_sample(model, processor, eval_dataset[i])
    f.write(json.dumps({"sequence_id": sid, "pages": parse(out)}) + "\n")
    print(i, sid)
f.close()

!python {DATA}/score.py --references /kaggle/working/val_labels.jsonl --predictions /kaggle/working/qwen_val.jsonl --output /kaggle/working/qwen_val_scores.json

0 seq_ec05f45d71c43f2c
1 seq_07838fb26cd1be54
2 seq_96839890d69d7698
3 seq_de542b63e6b19bc2
4 seq_d8bc3c6ac1bb6017
5 seq_84541a1f4b159018
6 seq_ee880b9612205860
7 seq_cd727990f22c777b
8 seq_4833e23eec54344b
9 seq_eaad741293da2ac1
10 seq_d8afceb0e8f9122d
11 seq_58858a2227cf3761
12 seq_c4448514b6de1ab9
13 seq_0ef2ec294597f572
14 seq_fc4c2b84b6ec32ff
15 seq_5f407d0064bb4a43
{
  "text_order_score": 0.1980701789500648,
  "balanced_joint_f1": 0.041600440792160034,
  "speaker_accuracy_on_matched": 0.20828760491715534,
  "joint_f1": 0.0676148608895748,
  "matched_token_coverage": 0.20831838418192647
}


In [14]:
# the 15 test sequences
f = open("/kaggle/working/qwen_test_predictions.jsonl", "w")
for sid in test_ids:
    out = generate_text_from_sample(model, processor, format_data(sid))
    f.write(json.dumps({"sequence_id": sid, "pages": parse(out)}) + "\n")
    print(sid)
f.close()

seq_6b18300406fa878f
seq_c780ac6d1dbd1242
seq_5303b549f62e9d3f
seq_d455cfd18edfb0ba
seq_e04ef3d20fd0baa8
seq_6aa313a0c0fa1c39
seq_32c47ca11b40ca8a
seq_81b1782990ab7950
seq_14469aeb38963085
seq_e9ffbb64217c2fed
seq_316bc7bd43ed6996
seq_cf7df70d4bba5da5
seq_e456a0394af99570
seq_c5622aca7487a661
seq_75aeca6c7131544b
